# Test `predict` on 1 claim
Cost-controlled smoke test: loads a single row from `claims.csv`, builds the prompt + image blocks, and calls the model once.

Requires credits on the Anthropic account for the API key in `.env`.

In [1]:
import sys, os
# run from repo root and make the code/ package importable
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, 'code'))

from loaders import load_claims
from images import build_image_blocks
from prompt import build_system_prompt, build_user_message
from model_client import predict

In [2]:
# 1 item only - keep cost minimal
claim = load_claims('/Users/xieqiqi/Learning/assignment/hackerrank-orchestrate-june26/dataset/sample_claims.csv')[0]
print(claim.user_id, '|', claim.claim_object, '|', len(claim.images), 'images')

image_blocks, missing = build_image_blocks(claim.images)
print('image blocks:', len(image_blocks), '| missing:', missing)

user_001 | car | 1 images
image blocks: 2 | missing: []


In [3]:
system = build_system_prompt()
user_content = build_user_message(
    claim,
    history_summary='Low-risk user with prior accepted vehicle claims.',
    evidence_text='The claimed object and relevant part should be clearly visible.',
    image_blocks=image_blocks,
)

In [4]:
result, usage = predict(system, user_content)
print('--- RESULT ---')
for k, v in result.model_dump().items():
    print(f'{k}: {v}')
print('--- USAGE ---')
print(usage)

--- RESULT ---
evidence_standard_met: True
evidence_standard_met_reason: The image clearly shows the rear of a car including the rear bumper area, trunk, and taillights, matching the claimed object and part.
risk_flags: [<RiskFlag.none: 'none'>]
issue_type: IssueType.dent
object_part: ObjectPart.rear_bumper
claim_status: ClaimStatus.supported
claim_status_justification: Image img_1 shows extensive damage to the rear of the white car: the trunk lid is dented and crumpled, the rear bumper cover is torn off exposing the structural reinforcement bar, and the body panels around the taillights are deformed. This is consistent with and exceeds the claimed dent in the rear bumper area.
supporting_image_ids: ['img_1']
valid_image: True
severity: Severity.high
--- USAGE ---
{'input': 613, 'output': 256, 'cache_read': 0, 'cache_write': 2259}
